## Imports

In [0]:
exec(open("/Workspace/Users/4020arjumandrashid@gmail.com/NASA-EONET/00_config").read())

In [0]:
%sql
USE CATALOG nasa_eonet;
USE SCHEMA events;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType
from pyspark.sql.functions import col, expr, to_timestamp, array_min, array_max, transform, from_json
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, DoubleType

In [0]:
!pip install geopandas
import geopandas as gpd

In [0]:
from pyspark.sql.functions import expr, broadcast

## Category

In [0]:
cat_df = spark.table(
    'raw_cat_table'
)

In [0]:
cat_df.show(5)

In [0]:
cat_df.dtypes

In [0]:
cat_df = cat_df.withColumn('id', col('id').cast('int'))

In [0]:
null_counts = cat_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in cat_df.columns])
null_counts.show()

In [0]:
cat_df.show(cat_df.count(), truncate=False)

In [0]:
silver_cat_df = cat_df.drop("ingestion_timestamp", "ingestion_id")

In [0]:
silver_cat_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_silver.categories")

## Sources

In [0]:
src_df = spark.table(
    'raw_src_table'
)

In [0]:
src_df.show(src_df.count(), truncate = False)

In [0]:
src_df.dtypes

In [0]:
null_count_src = src_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in src_df.columns])

null_count_src.show()

In [0]:
silver_src_df = src_df.drop('ingestion_id', 'ingestion_timestamp')

In [0]:
silver_src_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_silver.sources")

## Events

In [0]:
events_df = spark.table(
    'raw_events_table'
)

In [0]:
events_df.show(5)

In [0]:
events_df.dtypes

In [0]:
events_df.count()

In [0]:
null_counts_events =  events_df.select([sum(col(e).isNull().cast("int")) for e in events_df.columns])
display(null_counts_events)

In [0]:
# Count empty (non-null but zero-length) values per column.
# size() is a function in F, not a Column method. It only applies to array/map
# columns, so check emptiness by column type: empty array vs empty string.

def is_empty(c):
    if isinstance(events_df.schema[c].dataType, ArrayType):
        return F.size(col(c)) == 0
    return F.length(col(c)) == 0

empty_desc = events_df.select([
    sum(is_empty(e).cast("int")).alias(e) for e in events_df.columns
])
display(empty_desc)

Out of 1622 events, 954 have empty description fields.

In [0]:
# Count how many categories and sources each event belongs to.
# size() returns -1 for NULL arrays, so nulls are mapped to 0.
check_mapping = events_df.withColumn(
    "category_count",
    F.when(col("categories").isNull(), 0).otherwise(F.size("categories"))
).withColumn(
    "source_count",
    F.when(col("sources").isNull(), 0).otherwise(F.size("sources"))
)

# Check whether any category_count or source_count has a value > 1 or < 1.

print("More than one categories: ")
check_mapping.filter(
    (col("category_count") > 1) | (col("category_count") < 1)
).show(truncate=False)

print("More than one sources: ")
check_mapping.filter(
    (col("source_count") > 1) | (col("source_count") < 1)
).show(truncate=False)

# display(check_mapping)

Some events have multiple sources.

In [0]:
events_df = events_df.withColumn(
    'category_id',
    F.transform(
        F.col("categories"),
        lambda c: c.getItem("id")
    )
).withColumn(
    'source_id',
    F.transform(
        F.col("sources"),
        lambda s: s.getItem("id")
    )
)

In [0]:
events_df.show()

In [0]:
# using the struct to convert json string to array of date, geometry type and coords

geoms_schema = ArrayType(
    StructType([
        StructField("date", StringType()),
        StructField("type", StringType()),
        StructField("coordinates", ArrayType(DoubleType())),
    ])
)

silver_events_df = (
    events_df
    .withColumn("geoms", from_json(col("geometries"), geoms_schema))
    .withColumn(
        "min_date",
        array_min(transform(col("geoms"), lambda g: to_timestamp(g.getItem("date"))))
    )
    .withColumn(
        "max_date",
        array_max(transform(col("geoms"), lambda g: to_timestamp(g.getItem("date"))))
    )
    .withColumn(
        "geometry",
        expr(
            "transform(geoms, g -> st_geomfromgeojson("
            "to_json(named_struct('type', g.type, 'coordinates', g.coordinates))"
            "))"
        )
    )
    .drop("geoms")
)

silver_events_df.select("id", "source_id", "min_date", "max_date", "geometry").show(25, truncate=False)
silver_events_df.select("id", "source_id", "min_date", "max_date", "geometry").dtypes

In [0]:
silver_events_df = silver_events_df.withColumn("event_duration", F.datediff(F.col('max_date'), F.col('min_date')))

In [0]:
silver_events_df = silver_events_df.drop('categories', 'sources', 'geometries')

## Breakpoint for events - Geo-analysis

In [0]:
#silver_events_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_silver.events")

silver_events_df = spark.table(
    "nasa_eonet.events_silver.events"
)

In [0]:
silver_events_df.withColumn('geoemtries_count', F.size('geometry')).withColumn('distinct_geoemtries',
                                                                                F.size(F.array_distinct(F.expr("transform(geometry, g -> ST_AsText(g))")))).show(20)

In [0]:
from pyspark.sql.functions import expr

# For each event, average all point-geometry coordinates and build a single
# POINT rounded to 5 decimal places.
silver_events_df = (
    silver_events_df
    .withColumn(
        "avg_x",
        expr("aggregate(transform(geometry, g -> ST_X(g)), cast(0.0 as double), (acc, x) -> acc + x) / size(geometry)")
    )
    .withColumn(
        "avg_y",
        expr("aggregate(transform(geometry, g -> ST_Y(g)), cast(0.0 as double), (acc, y) -> acc + y) / size(geometry)")
    )
    .withColumn(
        "avg_point",
        expr("ST_Point(round(avg_x, 5), round(avg_y, 5))")
    )
    .drop("avg_x", "avg_y")
)

silver_events_df.select("id", "avg_point").show(20, truncate=False)

In [0]:
geojson_path = '/Volumes/nasa_eonet/events_silver/countriesgeojson/countries.geojson'
gdf = gpd.read_file(geojson_path)

In [0]:
gdf.head()

In [0]:
# Convert GeoPandas GeoDataFrame to a Spark DataFrame.
# Transfer geometries as WKT strings, then convert to Sedona geometries in Spark.
gdf_pd = gdf.copy()
gdf_pd['geometry_wkt'] = gdf_pd.geometry.apply(lambda g: g.wkt)
gdf_pd = gdf_pd.drop(columns=['geometry'])

countries_sdf = (
    spark.createDataFrame(gdf_pd)
    .withColumn("country_geom", expr("ST_GeomFromWKT(geometry_wkt)"))
    .filter(F.col("country_geom").isNotNull())
    # .withColumn(
    #     "country_geom",
    #     expr("CASE WHEN ST_IsValid(country_geom) THEN country_geom ELSE ST_MakeValid(country_geom) END")
    # )
    .select("name", "ISO3166-1-Alpha-3", "country_geom")      
)

In [0]:
countries_sdf.show(5)

In [0]:
silver_events_df = silver_events_df.join(
    countries_sdf,
    how = 'left',
    on = F.expr("ST_Within(avg_point, country_geom)")
)

In [0]:
silver_events_df.filter(silver_events_df['name'].isNotNull()).show(20)

In [0]:
print("Number of events where the country name has been identified: ", silver_events_df.filter(silver_events_df['name'].isNotNull()).count())
print("Number of events in the USA", silver_events_df.filter(silver_events_df['name'] == "United States of America").count())
print("Number of events outisde of the USA", silver_events_df.filter(silver_events_df['name'] != "United States of America").count())

In [0]:
print("Countries other that the USA:")
print(silver_events_df.filter(silver_events_df['name'] != 'United States of America').select('name').distinct().show())

In [0]:
silver_events_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_silver.events_geo")